# IBM Granite Speech ASR Evaluation

This notebook evaluates the IBM Granite Speech 4.1 ASR model (`ibm-granite/granite-speech-4.1-2b-plus`) on transcription accuracy.

In [ ]:
# @title Install packages and setup imports

# Standard Library Imports
import builtins
import json
import logging
import os
import struct
import sys
import traceback
import wave
from typing import Any, Dict, List, Optional, Tuple, Union

# Third-Party Imports
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
from google.cloud import storage
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

# The Magic Hack: Create a dummy class and inject it into Python's builtins 
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

# Add current directory dynamically to Python path to prevent ModuleNotFoundError
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

# Local Utility Imports
from common.audio_utils import preprocess_audio_for_model
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline
from common.prompts import COMMON_SYSTEM_PROMPT, COMMON_USER_INSTRUCTION

# Print active hardware validation status
if torch.cuda.is_available():
    print(f"CUDA is available: {torch.cuda.is_available()}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available inside the container environment.")
    print("You will only be able to execute the GCS-free Local CPU Sanity Check cell.")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# @title Constants
MODEL_NAME = "ibm-granite/granite-speech-4.1-2b-plus"
SELECTED_MODEL_KEY = "granite_speech_4_1_2b_plus"

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"

BATCH_SIZE = 16
LIMIT = 10

In [ ]:
# @title Load the model and processor

print(f"Loading Granite Speech model and processor: {MODEL_NAME}")
print(f"CUDA is available inside container: {torch.cuda.is_available()}")

# Dynamically select optimal precision based on hardware Tensor Core support (T4 vs A100)
optimal_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Optimal hardware precision dtype: {optimal_dtype}")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=optimal_dtype,
    trust_remote_code=True
)

print(f"Model loaded successfully on device: {model.device}")

In [ ]:
# @title Define helper functions for evaluation runner

def load_audio_torchaudio(
    path: str,
    sampling_rate: int = 16000,
    offset: float = 0.0,
    duration: Optional[float] = None,
) -> np.ndarray:
    """Loads and resamples only the requested segment frames natively using torchaudio."""
    info = torchaudio.info(path)
    sr = info.sample_rate

    frame_offset = int(offset * sr)
    num_frames = int(duration * sr) if duration else -1

    waveform, sr = torchaudio.load(path, frame_offset=frame_offset, num_frames=num_frames)

    if sr != sampling_rate:
        resampler = T.Resample(orig_freq=sr, new_freq=sampling_rate)
        waveform = resampler(waveform)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    return waveform.squeeze(0).numpy()

def prompt_formatter(
    entry: Optional[Dict[str, Any]],
    local_path: str,
) -> Tuple[str, str, Optional[Dict[str, Any]]]:
    """Constructs the formatted chat prompt and returns it along with the audio path and the entry dict."""
    ASR_PROMPT = f"<|audio|> {COMMON_USER_INSTRUCTION}"
    chat = [{"role": "system", "content": COMMON_SYSTEM_PROMPT}, {"role": "user", "content": ASR_PROMPT}]
    granite_chat_prompt = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return (granite_chat_prompt, local_path, entry)

def granite_inference(
    model: AutoModelForSpeechSeq2Seq,
    prompts: List[Tuple[str, str, Optional[Dict[str, Any]]]],
) -> Union[torch.Tensor, List[Union[torch.Tensor, str]]]:
    """Runs true parallel GPU batch inference using the model and processor with proper prompt formatting."""
    try:
        # 1. Load all audio file segments in the batch using torchaudio
        audios = []
        for p in prompts:
            chat_prompt, path, entry = p
            offset = entry.get("offset", 0.0) if entry else 0.0
            duration = entry.get("duration", None) if entry else None
            audio = load_audio_torchaudio(path, sampling_rate=16000, offset=offset, duration=duration)
            audios.append(audio)

        text_prompts = [p[0] for p in prompts]

        # 2. Process all audios together with their respective chat template prompts and padding enabled
        inputs = processor(
            audio=audios, 
            sampling_rate=16000, 
            text=text_prompts, 
            padding=True, 
            return_tensors="pt"
        )
        inputs.to(model.device, dtype=model.dtype)

        # 3. Generate tokens for the entire batch in a single parallel forward pass (limited to 40 tokens)
        outputs = model.generate(
            **inputs, 
            max_new_tokens=40,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id
        )
        # Slice out the input prompt tokens so only transcribed text is returned
        new_tokens = outputs[:, inputs["input_ids"].shape[-1]:]
        return new_tokens

    except Exception as e:
        logger.warning(f"Failed during parallel batch inference, falling back to sequential: {e}")
        # Fallback to sequential processing if the batch fails due to memory or padding errors
        outputs = []
        for p in prompts:
            try:
                chat_prompt, path, entry = p
                offset = entry.get("offset", 0.0) if entry else 0.0
                duration = entry.get("duration", None) if entry else None
                audio = load_audio_torchaudio(path, sampling_rate=16000, offset=offset, duration=duration)

                inputs = processor(audio=audio, sampling_rate=16000, text=chat_prompt, return_tensors="pt")
                inputs.to(model.device, dtype=model.dtype)
                out = model.generate(
                    **inputs, 
                    max_new_tokens=40,
                    eos_token_id=processor.tokenizer.eos_token_id,
                    pad_token_id=processor.tokenizer.pad_token_id
                )
                new_tokens = out[0, inputs["input_ids"].shape[-1]:]
                outputs.append(new_tokens)
            except Exception as ex:
                logger.error(f"Failed during sequential fallback for {p[1]}: {ex}")
                outputs.append("[ERROR]")
        return outputs

def result_decoder(
    ans: Union[torch.Tensor, str],
    model: AutoModelForSpeechSeq2Seq,
) -> str:
    """Extracts the transcription safely using the processor."""
    if isinstance(ans, str) and ans == "[ERROR]":
        return ""
    try:
        # ans is already a 1D token sequence tensor, decode it directly
        return processor.decode(ans, skip_special_tokens=True)
    except Exception as e:
        logger.error(f"Decoding failed: {e}")
        return ""

In [ ]:
# @title Local ASR CPU Sanity Check (GCS-Free)

try:
    print("--- Phase 1: Creating Local Dummy Audio File ---")
    dummy_input = "/tmp/temp_silent_raw.wav"
    dummy_output = "/tmp/temp_silent_preprocessed.wav"

    # Create a 1-second silent mono WAV file at 8kHz inside container's native /tmp/
    with wave.open(dummy_input, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2) # 16-bit audio
        w.setframerate(8000)
        w.writeframes(struct.pack("<h", 0) * 8000)
    print(f"Created dummy input: {dummy_input}")

    print("\n--- Phase 2: Testing torchaudio Preprocessing Utility ---")
    # Preprocess using our updated torchaudio helper (resamples to 16kHz)
    success = preprocess_audio_for_model(dummy_input, dummy_output, target_sr=16000)
    if not success:
        raise RuntimeError("Audio preprocessing failed!")

    print("\n--- Phase 3: Testing Batch Inference Loop ---")
    prompt = prompt_formatter(None, dummy_output)
    outputs = granite_inference(model, [prompt])
    print(f"Inference successful! Outputs list length: {len(outputs)}")
    print(f"Outputs[0] type: {type(outputs[0])}")

    print("\n--- Phase 4: Decoding Final Transcription ---")
    transcription = result_decoder(outputs[0], model)
    print("\n--- Sanity Check Complete! ---")
    print(f"Predicted Transcript: '{transcription.strip()}'")

except Exception:
    print("\n❌ CRITICAL ERROR OCCURRED:")
    traceback.print_exc()

finally:
    # Cleanup local temp files
    for path in [dummy_input, dummy_output]:
        if os.path.exists(path):
            os.remove(path)

In [ ]:
# @title Run Evaluation

# Hard gatekeeper: Ensure GPU is active before starting the heavy parallel evaluation run
assert torch.cuda.is_available(), (
    "CUDA is not available! The parallel batch evaluation runner requires GPU hardware acceleration. "
    "Please make sure GPU reservations are enabled in your active container deployment."
)

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Sort manifest entries by duration to minimize padding silence and accelerate batching!
manifest_data = sorted(manifest_data, key=lambda x: x.get("duration", 0.0))

# Run the standardized evaluation pipeline using the model-agnostic runner
results_list = run_inference_pipeline(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=granite_inference,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    preprocess_fn=None
)

In [ ]:
# @title Upload results directly to GCS from memory
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    SELECTED_MODEL_KEY, 
    EXPERIMENT_NAME, 
    results_list
)